# Gemma 7B LoRA Fine-Tuning for Implicit Sentiment Analysis (ISA)

This notebook performs **Standard Fine-tuning** for the Implicit Sentiment Analysis task on the **Laptop** and **Restaurant** datasets.

**Configuration:**
- **Model:** Google Gemma 7B (BFloat16)
- **Hardware:** NVIDIA A100 (Google Colab)
- **Method:** LoRA (Low-Rank Adaptation)
- **Libraries:** Hugging Face `transformers`, `peft`, `trl`

### Pipeline:
1. Install dependencies & configure environment.
2. Parse SemEval-style XML data (Laptop & Restaurant).
3. Define unified training function with LoRA.
4. Fine-tune separately for each domain.
5. Run inference on test sets.
6. Evaluate and save metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 1. Install Dependencies
!pip uninstall -y torch torchvision torchaudio tensorflow fastai
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers peft bitsandbytes trl accelerate

In [ ]:
# @title 2. Imports & Configuration
import os
import torch
import xml.etree.ElementTree as ET
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

# @markdown **Enter your HF Token (requires Gemma access):**
HF_TOKEN = "" # @param {type:"string"}
login(token=HF_TOKEN)

logging.set_verbosity_error()

In [ ]:
# @title 3. Define Paths & Hyperparameters

# @markdown **Path to training XML files:**
LAPTOP_TRAIN_XML = "/content/drive/MyDrive/ATAC2026/data/laptop/laptop_train_implicit-labeled.xml" # @param {type:"string"}
RESTAURANT_TRAIN_XML = "/content/drive/MyDrive/ATAC2026/data/restaurant/restaurant_train_implicit-labeled.xml" # @param {type:"string"}

# @markdown **Base model name or path:**
MODEL_ID = "google/gemma-7b" # @param {type:"string"}

# @markdown **Output directory for saved models:**
OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/fine_tuned_models" # @param {type:"string"}

# Hyperparameters
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.1
BATCH_SIZE = 8
GRAD_ACC_STEPS = 2
LEARNING_RATE = 2e-4
NUM_EPOCHS = 4
MAX_SEQ_LENGTH = 1024
LR_SCHEDULER = "cosine"

In [ ]:
# @title 4. Data Processing

def parse_semeval_xml(xml_path):
    """Parse SemEval-style XML into a DataFrame of (sentence, aspect, sentiment) tuples."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    data = []

    for sentence in root.findall('sentence'):
        text = sentence.find('text').text
        aspect_terms = sentence.find('aspectTerms')

        if aspect_terms is not None:
            for aspect in aspect_terms.findall('aspectTerm'):
                term = aspect.get('term')
                polarity = aspect.get('polarity')
                implicit = aspect.get('implicit_sentiment')

                if polarity == 'conflict':
                    continue

                data.append({
                    'text': text,
                    'aspect': term,
                    'sentiment': polarity,
                    'is_implicit': implicit
                })

    return pd.DataFrame(data)

def format_prompt(sample):
    """Format a sample into instruction-tuning prompt for Gemma."""
    instruction = "Identify the sentiment polarity (positive, negative, or neutral) of the specific aspect in the following sentence."
    input_text = f"Sentence: {sample['text']}\nAspect: {sample['aspect']}"
    response = sample['sentiment']
    full_prompt = f"Instruction:\n{instruction}\n\nInput:\n{input_text}\n\nResponse:\n{response}<eos>"
    return {"text": full_prompt}

# Verify data loading
if os.path.exists(LAPTOP_TRAIN_XML):
    df_laptop = parse_semeval_xml(LAPTOP_TRAIN_XML)
    print(f"Loaded {len(df_laptop)} samples from Laptop.")
    print("Example:", df_laptop.iloc[0].to_dict())
else:
    print(f"Warning: {LAPTOP_TRAIN_XML} not found. Please upload the file.")

In [ ]:
# @title 5. Training Function
import shutil

def run_fine_tuning(dataset_name, xml_path, output_subfolder):
    print(f"\n{'='*20}\nStarting Fine-tuning for: {dataset_name}\n{'='*20}")

    # Prepare data
    df = parse_semeval_xml(xml_path)
    dataset = Dataset.from_pandas(df)
    dataset = dataset.map(format_prompt)
    dataset = dataset.select_columns(['text'])
    print(f"Training samples: {len(dataset)}")

    # Load model (BFloat16, no quantization)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map={"": 0}
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # LoRA configuration
    peft_config = LoraConfig(
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        r=LORA_R,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    # Training arguments (save checkpoints to local temp, then copy final to Drive)
    local_temp_dir = f"/content/temp_training/{output_subfolder}"
    training_args = SFTConfig(
        output_dir=local_temp_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        learning_rate=LEARNING_RATE,
        fp16=False,
        bf16=True,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        group_by_length=True,
        lr_scheduler_type=LR_SCHEDULER,
        max_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        packing=False
    )

    # Train
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
    )
    trainer.train()

    # Save final model to Drive
    final_drive_path = os.path.join(OUTPUT_DIR, output_subfolder)
    os.makedirs(final_drive_path, exist_ok=True)
    trainer.model.save_pretrained(final_drive_path)
    tokenizer.save_pretrained(final_drive_path)

    local_runs_dir = os.path.join(local_temp_dir, "runs")
    if os.path.exists(local_runs_dir):
        shutil.copytree(local_runs_dir, os.path.join(final_drive_path, "runs"), dirs_exist_ok=True)

    print(f"Model saved to: {final_drive_path}")

    # Cleanup
    del model, trainer
    torch.cuda.empty_cache()
    shutil.rmtree(local_temp_dir, ignore_errors=True)

    return final_drive_path

In [ ]:
# @title 6. Fine-tune Laptop
if os.path.exists(LAPTOP_TRAIN_XML):
    laptop_model_path = run_fine_tuning("Laptop", LAPTOP_TRAIN_XML, "gemma-7b-laptop-isa-lora-end")
else:
    print("Please upload the Laptop XML file and verify the path.")

In [ ]:
# @title 7. Fine-tune Restaurant
if os.path.exists(RESTAURANT_TRAIN_XML):
    restaurant_model_path = run_fine_tuning("Restaurant", RESTAURANT_TRAIN_XML, "gemma-7b-restaurant-isa-lora-end")
else:
    print("Please upload the Restaurant XML file and verify the path.")

In [ ]:
# @title 8. Run Inference on Test Sets

# @markdown **Path to test XML files:**
LAPTOP_TEST_XML = "/content/drive/MyDrive/ATAC2026/data/laptop/laptop_test_implicit-labeled.xml" # @param {type:"string"}
RESTAURANT_TEST_XML = "/content/drive/MyDrive/ATAC2026/data/restaurant/restaurant_test_implicit-labeled.xml" # @param {type:"string"}

# @markdown **Output directory for inference results:**
INFERENCE_OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/inference_results" # @param {type:"string"}
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

# Auto-resolve model paths if not set by training cells
if 'laptop_model_path' not in globals():
    laptop_model_path = os.path.join(OUTPUT_DIR, "gemma-7b-laptop-isa-lora-end")
if 'restaurant_model_path' not in globals():
    restaurant_model_path = os.path.join(OUTPUT_DIR, "gemma-7b-restaurant-isa-lora-end")

from tqdm import tqdm

def run_inference_on_dataset(test_xml_path, model_path, base_model_id, dataset_name):
    """Run inference on the full test set and save predictions to CSV."""
    print(f"Loading test data from {test_xml_path}...")
    df_test = parse_semeval_xml(test_xml_path)
    print(f"Found {len(df_test)} test samples.")

    print(f"Loading model from {model_path}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        torch_dtype=torch.bfloat16,
        device_map={"": 0}
    )
    model = PeftModel.from_pretrained(base_model, model_path)
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    tokenizer.padding_side = "left"

    # Build prompts
    inputs = []
    for _, row in df_test.iterrows():
        instruction = "Identify the sentiment polarity (positive, negative, or neutral) of the specific aspect in the following sentence."
        input_text = f"Sentence: {row['text']}\nAspect: {row['aspect']}"
        prompt = f"Instruction:\n{instruction}\n\nInput:\n{input_text}\n\nResponse:\n"
        inputs.append(prompt)

    # Batch inference
    batch_size = 16
    preds = []

    print("Running inference...")
    for i in tqdm(range(0, len(inputs), batch_size)):
        batch_prompts = inputs[i:i+batch_size]
        batch_encodings = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **batch_encodings,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        decoded_outputs = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for out in decoded_outputs:
            response = out.split("Response:\n")[-1].strip().lower()
            for sentiment in ['positive', 'negative', 'neutral']:
                if response.startswith(sentiment):
                    response = sentiment
                    break
            preds.append(response)

    df_test['prediction'] = preds

    output_csv_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{dataset_name}_inference_results_end.csv")
    df_test.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Saved results to: {output_csv_path}")

    del model, base_model
    torch.cuda.empty_cache()
    return df_test

df_laptop_results = None
df_restaurant_results = None

if os.path.exists(LAPTOP_TEST_XML) and os.path.exists(laptop_model_path):
    print(">>> Laptop Inference...")
    df_laptop_results = run_inference_on_dataset(LAPTOP_TEST_XML, laptop_model_path, MODEL_ID, "Laptop")
else:
    print("Skipping Laptop test (file not found or model not trained).")

if os.path.exists(RESTAURANT_TEST_XML) and os.path.exists(restaurant_model_path):
    print("\n>>> Restaurant Inference...")
    df_restaurant_results = run_inference_on_dataset(RESTAURANT_TEST_XML, restaurant_model_path, MODEL_ID, "Restaurant")
else:
    print("Skipping Restaurant test (file not found or model not trained).")

In [ ]:
# @title 9. Evaluation Metrics
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import os

def get_config_info():
    config_info = f"==== CONFIGURATION ====\n"
    config_info += f"Model: {MODEL_ID} (Full BF16)\n"
    config_info += f"LoRA Rank (r): {LORA_R}\n"
    config_info += f"LoRA Alpha: {LORA_ALPHA}\n"
    config_info += f"LoRA Dropout: {LORA_DROPOUT}\n"
    config_info += f"Epochs: {NUM_EPOCHS}\n"
    config_info += f"Max Seq Length: {MAX_SEQ_LENGTH}\n"
    config_info += f"Learning Rate: {LEARNING_RATE}\n"
    config_info += f"Scheduler: {LR_SCHEDULER}\n"
    config_info += f"Batch Size: {BATCH_SIZE}\n"
    config_info += "=" * 40 + "\n\n"
    return config_info

def evaluate_and_save(df_results, dataset_name, output_file_path):
    """Compute Overall, ESA, and ISA metrics, then save the report to a file."""
    if df_results is None:
        print(f"No results for {dataset_name}. Skipping.")
        return

    df_results['is_implicit'] = df_results['is_implicit'].map(lambda x: str(x).lower() == 'true')
    y_true = df_results['sentiment'].str.lower()
    y_pred = df_results['prediction'].str.lower()

    valid_labels = ['positive', 'negative', 'neutral']
    mask = y_true.isin(valid_labels) & y_pred.isin(valid_labels)
    df_filtered = df_results[mask].copy()

    y_true = df_filtered['sentiment'].str.lower()
    y_pred = df_filtered['prediction'].str.lower()
    is_implicit = df_filtered['is_implicit']

    # Overall
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')

    # Explicit Subset (ESA)
    mask_explicit = ~is_implicit
    acc_explicit = accuracy_score(y_true[mask_explicit], y_pred[mask_explicit]) if mask_explicit.sum() > 0 else 0.0
    f1_explicit = f1_score(y_true[mask_explicit], y_pred[mask_explicit], average='macro') if mask_explicit.sum() > 0 else 0.0

    # Implicit Subset (ISA)
    mask_implicit = is_implicit
    acc_implicit = accuracy_score(y_true[mask_implicit], y_pred[mask_implicit]) if mask_implicit.sum() > 0 else 0.0
    f1_implicit = f1_score(y_true[mask_implicit], y_pred[mask_implicit], average='macro') if mask_implicit.sum() > 0 else 0.0

    report = get_config_info()
    report += f"==== EVALUATION REPORT: {dataset_name} ====\n"
    report += f"Total Samples: {len(df_filtered)}\n"
    report += f"Explicit Samples (ESA): {mask_explicit.sum()}\n"
    report += f"Implicit Samples (ISA): {mask_implicit.sum()}\n\n"
    report += f"1. Overall Performance:\n"
    report += f"   - Accuracy: {acc:.4f}\n"
    report += f"   - F1-Macro: {f1_macro:.4f}\n"
    report += f"2. Explicit Subset Performance (ESA):\n"
    report += f"   - Accuracy (ESA): {acc_explicit:.4f}\n"
    report += f"   - F1-Macro (ESA): {f1_explicit:.4f}\n"
    report += f"3. Implicit Subset Performance (ISA):\n"
    report += f"   - Accuracy (ISA): {acc_implicit:.4f}\n"
    report += f"   - F1-Macro (ISA): {f1_implicit:.4f}\n"
    report += "\n" + "=" * 40 + "\n"

    print(report)
    with open(output_file_path, "w") as f:
        f.write(report)
    print(f"Results saved to: {output_file_path}")

# @markdown **Output directories:**
EVAL_OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/evaluation_results" # @param {type:"string"}
INFERENCE_OUTPUT_DIR = "/content/drive/MyDrive/ATAC2026/inference_results" # @param {type:"string"}
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

def load_results_if_missing(df, dataset_name):
    """Load inference results from CSV if the DataFrame is not in memory."""
    if df is not None:
        return df
    csv_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{dataset_name}_inference_results_end.csv")
    if os.path.exists(csv_path):
        print(f"Loading {dataset_name} results from CSV: {csv_path}")
        return pd.read_csv(csv_path)
    print(f"No results found for {dataset_name}.")
    return None

current_laptop_df = df_laptop_results if 'df_laptop_results' in globals() else None
current_restaurant_df = df_restaurant_results if 'df_restaurant_results' in globals() else None

df_laptop_final = load_results_if_missing(current_laptop_df, "Laptop")
if df_laptop_final is not None:
    evaluate_and_save(df_laptop_final, "Laptop", os.path.join(EVAL_OUTPUT_DIR, "laptop_metrics_end.txt"))

df_restaurant_final = load_results_if_missing(current_restaurant_df, "Restaurant")
if df_restaurant_final is not None:
    evaluate_and_save(df_restaurant_final, "Restaurant", os.path.join(EVAL_OUTPUT_DIR, "restaurant_metrics_end.txt"))